# Download and convert raw direct-detection data to JSON

Run this notebook in Google Colab. It downloads supported files from `recent_data/raw`, extracts numeric x-y pairs, and saves them as JSON.

In [ ]:
from pathlib import Path
from decimal import Decimal, InvalidOperation
import csv, json, re, subprocess
import requests
try:
    import openpyxl
except ImportError:
    %pip -q install openpyxl
    import openpyxl
OWNER, REPO, REF = 'BrownParticleAstro', 'dmtools_notebooks', 'main'
RAW_DIR = 'recent_data/raw'
WORKDIR = Path('/content/dmtools_raw_downloads')
WORKDIR.mkdir(parents=True, exist_ok=True)
SUPPORTED = {'.csv', '.txt', '.xlsx', '.xlsm'}
print('Ready. Download directory:', WORKDIR)

## Download supported files

The GitHub Contents API discovers filenames; `wget` downloads the files.

In [ ]:
api_url = f'https://api.github.com/repos/{OWNER}/{REPO}/contents/{RAW_DIR}?ref={REF}'
response = requests.get(api_url, timeout=30)
response.raise_for_status()
entries = response.json()
if not isinstance(entries, list):
    raise RuntimeError(f'Expected a directory listing, got: {entries}')
data_files = [e for e in entries if e.get('type') == 'file' and Path(e.get('name', '')).suffix.lower() in SUPPORTED]
if not data_files:
    raise RuntimeError('No supported CSV/TXT/Excel files found.')
for entry in data_files:
    name = entry['name']
    url = entry.get('download_url') or f'https://raw.githubusercontent.com/{OWNER}/{REPO}/{REF}/{RAW_DIR}/{name}'
    print('Downloading', name)
    subprocess.run(['wget', '--quiet', '--show-progress', '-O', str(WORKDIR / name), url], check=True)
print(f'\nDownloaded {len(data_files)} supported files.')

## Parse table sections

Numeric tokens are retained as strings to preserve their original notation. Blank lines and headers separate sections.

In [ ]:
NUMBER_RE = re.compile(r'^[+-]?(?:(?:\d+(?:\.\d*)?)|(?:\.\d+))(?:[eEdD][+-]?\d+)?$')
def number_text(value):
    if value is None or isinstance(value, bool):
        return None
    text = str(value).strip()
    if not text or not NUMBER_RE.fullmatch(text):
        return None
    try:
        Decimal(text.replace('D', 'E').replace('d', 'e'))
    except InvalidOperation:
        return None
    return text
def row_to_cells(line, suffix):
    if suffix.lower() == '.csv' or ',' in line:
        return next(csv.reader([line], skipinitialspace=True))
    return line.split()
def parse_rows(rows):
    sections, pairs, label = [], [], None
    def flush():
        nonlocal pairs
        if pairs:
            sections.append({'label': label or f'section_{len(sections) + 1}', 'pairs': pairs})
        pairs = []
    for raw_row in rows:
        cells = ['' if c is None else str(c).strip() for c in raw_row]
        if not any(cells):
            flush()
            label = None
            continue
        numeric = [number_text(c) for c in cells]
        numeric = [v for v in numeric if v is not None]
        if len(numeric) >= 2:
            pairs.append(numeric[-2:])
        else:
            flush()
            header = [c for c in cells if c]
            if header:
                label = ','.join(header)
    flush()
    return sections
def parse_text_file(path):
    rows = []
    with path.open('r', encoding='utf-8-sig', errors='replace') as stream:
        for line in stream:
            line = line.strip()
            if line and not line.startswith('#'):
                rows.append(row_to_cells(line, path.suffix))
    return parse_rows(rows)
def parse_excel_file(path):
    workbook = openpyxl.load_workbook(path, read_only=True, data_only=True)
    sections = []
    try:
        for sheet in workbook.worksheets:
            rows = [[str(v).strip() if v is not None else '' for v in row] for row in sheet.iter_rows(values_only=True)]
            for section in parse_rows(rows):
                section['label'] = f'{sheet.title}: {section["label"]}'
                sections.append(section)
    finally:
        workbook.close()
    return sections
def parse_file(path):
    if path.suffix.lower() in {'.xlsx', '.xlsm'}:
        return parse_excel_file(path)
    if path.suffix.lower() in {'.csv', '.txt'}:
        return parse_text_file(path)
    raise ValueError(f'Unsupported file type: {path.name}')

## Convert files and download the JSON result

In [ ]:
results, errors = {}, {}
for path in sorted(WORKDIR.iterdir()):
    if path.suffix.lower() not in SUPPORTED:
        continue
    try:
        sections = parse_file(path)
        if sections:
            results[path.name] = sections
            print(f'{path.name}: {sum(len(s["pairs"]) for s in sections)} points in {len(sections)} section(s)')
        else:
            errors[path.name] = 'No numeric two-column table detected'
    except Exception as exc:
        errors[path.name] = f'{type(exc).__name__}: {exc}'
output_path = Path('/content/raw_data.json')
with output_path.open('w', encoding='utf-8') as stream:
    json.dump(results, stream, ensure_ascii=False, indent=2)
print(f'\nWrote {output_path} ({output_path.stat().st_size:,} bytes)')
if errors:
    print(json.dumps(errors, indent=2))
from google.colab import files
files.download(str(output_path))